# DGE & DBI — Construcción de matrices
**Pasos:**,
1. Leer los CSV de `Gene_Expression_reduced/` y extraer identificadores ENSG.

2. Transformar ENSG → símbolo génico usando `archivo.csv`.

3. Calcular la matriz de similitud semántica GO (DBI) con `compute_gene_similarity_matrix_go3`.

4. Guardar `{dataset}_DGE.csv` (correlación Pearson, re-etiquetada con símbolos) y `{dataset}_DBI.csv` (similitud GO3), con genes en el mismo orden.

In [1]:
##################################
# Imports
##################################
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import dataclass
from typing import List, Literal, Sequence, Tuple, Union
import warnings
warnings.filterwarnings("ignore")

try:
    import go3
    GO3_AVAILABLE = True
except ImportError:
    GO3_AVAILABLE = False
    print("go3 no está instalado. Instálalo con: pip install go3")

##################################
# Configuración de rutas
##################################
BASE_DIR     = Path(r"C:\Users\benja\Desktop\workspace\Thesis\Datasets")
GE_DIR       = BASE_DIR / "Gene_Expression_reduced"
MAPPING_FILE = BASE_DIR / "archivo.csv"
OBO_PATH     = BASE_DIR / "Bio_Info Resources" / "go.obo"
GAF_PATH     = BASE_DIR / "Bio_Info Resources" / "goa_human.gaf"
OUTPUT_DIR   = BASE_DIR / "DGE_DBI"
OUTPUT_DIR.mkdir(exist_ok=True)

##################################
# Tipos y opciones go3
##################################
Ontology          = Literal["BP", "MF", "CC"]
Groupwise         = Literal["bma", "max", "avg", "hausdorff", "simgic"]
SimilarityMeasure = Literal["resnik", "lin", "jc", "simrel", "iccoef", "graphic", "wang", "topoicsim"]
DistanceTransform = Literal["auto", "one_minus", "max_minus", "reciprocal"]

@dataclass(frozen=True)
class GeneSimilarityOptions:
    ontology:        Ontology          = "BP"
    measure:         SimilarityMeasure = "wang"
    groupwise:       Groupwise         = "bma"
    distance_method: DistanceTransform = "auto"
    load_go_terms:   bool              = True
    num_threads_go3: int               = 0    # 0 = auto

In [2]:
##################################
# Funciones auxiliares
##################################

def cargar_mapping(mapping_file: Path) -> pd.DataFrame:
    """
    Carga archivo.csv con el mapeo ENSG → símbolo génico.
    Asume columnas 'ensembl_id' y 'symbol' (o las detecta automáticamente).

    Retorna
    -------
    pd.DataFrame con columnas normalizadas: ensembl_id, symbol
    """
    df = pd.read_csv(mapping_file, dtype=str).fillna("")

    # Detectar columnas automáticamente si no tienen el nombre exacto
    cols = [c.lower().strip() for c in df.columns]
    df.columns = cols

    # Mapeo flexible de nombres de columna
    rename = {}
    for c in cols:
        if "ensembl" in c or c.startswith("ensg"):
            rename[c] = "ensembl_id"
        elif "symbol" in c or "gene" in c or "name" in c:
            rename[c] = "symbol"
    df = df.rename(columns=rename)

    if "ensembl_id" not in df.columns or "symbol" not in df.columns:
        raise ValueError(
            f"No se encontraron columnas 'ensembl_id' y 'symbol' en {mapping_file}.\n"
            f"Columnas disponibles: {list(df.columns)}"
        )

    # Limpiar versiones de ID (ENSG00000123456.3 → ENSG00000123456)
    df["ensembl_id"] = df["ensembl_id"].str.split(".").str[0].str.strip()
    df["symbol"]     = df["symbol"].str.strip()

    # Eliminar duplicados: quedarse con el primer símbolo por ENSG
    df = df.drop_duplicates(subset="ensembl_id", keep="first")

    print(f"  Mapping cargado : {len(df):,} genes  ({mapping_file.name})")
    return df[["ensembl_id", "symbol"]].reset_index(drop=True)


def ensg_a_simbolo(ensg_list: List[str],
                   mapping_df: pd.DataFrame) -> Tuple[List[str], List[str]]:
    """
    Convierte una lista de IDs ENSG a símbolos génicos usando el mapping local.
    Los IDs sin símbolo en el mapping se conservan como fallback.

    Retorna
    -------
    simbolos      : lista de símbolos (mismo largo que ensg_list)
    sin_mapeo     : IDs que no tenían símbolo en el mapping
    """
    mapa      = mapping_df.set_index("ensembl_id")["symbol"].to_dict()
    simbolos  = []
    sin_mapeo = []

    for ensg in ensg_list:
        ensg_base = ensg.split(".")[0]          # eliminar versión si la hay
        sym = mapa.get(ensg_base, "")
        if sym:
            simbolos.append(sym)
        else:
            simbolos.append(ensg_base)          # fallback al propio ID
            sin_mapeo.append(ensg_base)

    return simbolos, sin_mapeo


def compute_gene_similarity_matrix_go3(
    genes: Sequence[str],
    *,
    obo_path: Union[str, Path],
    gaf_path: Union[str, Path],
    go3_opts: GeneSimilarityOptions = GeneSimilarityOptions(),
) -> Tuple[List[str], np.ndarray]:
    """
    Calcula la matriz de similitud semántica génica usando go3.

    Parámetros
    ----------
    genes    : lista de símbolos génicos
    obo_path : ruta al archivo .obo de Gene Ontology
    gaf_path : ruta al archivo GAF de anotaciones
    go3_opts : opciones de configuración de go3

    Retorna
    -------
    (ordered_genes, similarity_matrix)
    """
    if not GO3_AVAILABLE:
        raise RuntimeError("go3 no está instalado. Instálalo con: pip install go3")
    if len(genes) == 0:
        raise ValueError("genes debe ser una lista no vacía.")
    if go3_opts.ontology not in ("BP", "MF", "CC"):
        raise ValueError("ontology debe ser 'BP', 'MF' o 'CC'.")

    go3.set_num_threads(int(go3_opts.num_threads_go3))

    if go3_opts.load_go_terms:
        go3.load_go_terms(str(obo_path))

    annotations = go3.load_gaf(str(gaf_path))
    counter     = go3.build_term_counter(annotations)

    ordered, dist = go3.gene_distance_matrix(
        genes,
        ontology           = go3_opts.ontology,
        similarity         = go3_opts.measure,
        groupwise          = go3_opts.groupwise,
        counter            = counter,
        distance_transform = go3_opts.distance_method,
    )

    sim = 1.0 - np.array(dist, dtype=np.float64)
    np.fill_diagonal(sim, 1.0)
    sim = np.clip(sim, 0.0, 1.0)

    return list(ordered), sim

In [5]:
##################################
# PASO 1: Cargar mapping ENSG → símbolo
##################################
mapping_df = cargar_mapping(MAPPING_FILE)

  Mapping cargado : 58,735 genes  (archivo.csv)


In [6]:
##################################
# PASO 2: Leer CSVs post-PCA y construir DGE + DBI por dataset
##################################

# Opciones go3 (ajustar según necesidad)
GO3_OPTS = GeneSimilarityOptions(
    ontology        = "BP",    # Biological Process
    measure         = "wang",  # Medida de Wang (no requiere IC externo)
    groupwise       = "bma",   # Best Match Average
    distance_method = "auto",
    load_go_terms   = True,
    num_threads_go3 = 0        # 0 = todos los cores disponibles
)

RESULTADOS = {}   # { dataset_name: { 'DGE': df, 'DBI': df, 'genes': [...] } }

csv_files = sorted(GE_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No se encontraron archivos CSV en {GE_DIR}")

print(f"Datasets encontrados: {len(csv_files)}")

for csv_path in csv_files:
    nombre = csv_path.stem
    print(f"\n{'='*60}")
    print(f"  Dataset : {nombre}")
    print(f"{'='*60}")

    # ── Leer CSV post-PCA ─────────────────────────────────────
    df_raw     = pd.read_csv(csv_path, index_col=0, dtype=str)
    cols_genes = [c for c in df_raw.columns if c not in ["ID", "class"]]

    # Extraer solo expresión (float) e IDs ENSG
    dataX  = df_raw[cols_genes].astype(float).values    # (muestras, genes)
    ensg_ids = np.array(cols_genes)

    print(f"  Muestras : {dataX.shape[0]:,}")
    print(f"  Genes    : {dataX.shape[1]:,}")

    # ── PASO 2a: Convertir ENSG → símbolo ────────────────────
    simbolos, sin_mapeo = ensg_a_simbolo(list(ensg_ids), mapping_df)

    if sin_mapeo:
        print(f"  ⚠  {len(sin_mapeo):,} genes sin símbolo → se usa ID ENSG como fallback")

    # Manejar símbolos duplicados añadiendo sufijo numérico
    seen   = {}
    simbolos_uniq = []
    for s in simbolos:
        if s not in seen:
            seen[s] = 0
            simbolos_uniq.append(s)
        else:
            seen[s] += 1
            simbolos_uniq.append(f"{s}_{seen[s]}")
    simbolos = simbolos_uniq

    # ── PASO 2b: Calcular DGE (correlación Pearson normalizada) ──
    print("  Calculando DGE (Pearson)...")
    corr_matrix = np.corrcoef(dataX, rowvar=False)

    # Sanidad numérica
    corr_matrix = (corr_matrix + corr_matrix.T) / 2
    np.fill_diagonal(corr_matrix, 1.0)
    corr_matrix = np.clip(corr_matrix, -1.0, 1.0)

    # Normalizar [-1, 1] → [0, 1]
    corr_norm = (corr_matrix + 1) / 2

    df_dge = pd.DataFrame(corr_norm, index=simbolos, columns=simbolos)

    # ── PASO 2c: Calcular DBI (similitud semántica GO) ───────
    print("  Calculando DBI (go3)...")
    ordered_genes, sim_matrix = compute_gene_similarity_matrix_go3(
        simbolos,
        obo_path  = OBO_PATH,
        gaf_path  = GAF_PATH,
        go3_opts  = GO3_OPTS,
    )

    # go3 puede reordenar los genes → alinear con el orden de DGE
    orden_dbi = [simbolos.index(g) if g in simbolos else -1
                 for g in ordered_genes]
    orden_dbi = [i for i in orden_dbi if i >= 0]

    # Reindexar DBI al orden de DGE
    sim_alineada = np.zeros((len(simbolos), len(simbolos)))
    for i_dbi, i_dge in enumerate(orden_dbi):
        for j_dbi, j_dge in enumerate(orden_dbi):
            sim_alineada[i_dge, j_dge] = sim_matrix[i_dbi, j_dbi]

    df_dbi = pd.DataFrame(sim_alineada, index=simbolos, columns=simbolos)

    # ── PASO 2d: Guardar DGE y DBI ───────────────────────────
    path_dge = OUTPUT_DIR / f"{nombre}_DGE.csv"
    path_dbi = OUTPUT_DIR / f"{nombre}_DBI.csv"

    df_dge.to_csv(path_dge)
    df_dbi.to_csv(path_dbi)

    print(f"  DGE guardado en : {path_dge}")
    print(f"  DBI guardado en : {path_dbi}")

    # Stats resumen
    tri_dge = corr_norm[np.triu_indices_from(corr_norm, k=1)]
    tri_dbi = sim_alineada[np.triu_indices_from(sim_alineada, k=1)]

    print(f"  DGE — media: {tri_dge.mean():.4f}  std: {tri_dge.std():.4f}")
    print(f"  DBI — media: {tri_dbi.mean():.4f}  std: {tri_dbi.std():.4f}")
    print(f"  DBI — genes con anotación GO: "
          f"{len([g for g in ordered_genes if g in simbolos]):,} / {len(simbolos):,}")

    RESULTADOS[nombre] = {
        'DGE'    : df_dge,
        'DBI'    : df_dbi,
        'genes'  : simbolos,
        'ensg'   : list(ensg_ids),
    }

Datasets encontrados: 5

  Dataset : GSE40419_corr_pearson_norm
  Muestras : 351
  Genes    : 351
  ⚠  31 genes sin símbolo → se usa ID ENSG como fallback
  Calculando DGE (Pearson)...
  Calculando DBI (go3)...


ValueError: Genes not found in mapping: BVES, LINC00467, FAM86JP, MFSD4B, LINC02880, NINJ2-AS1, ZBED10P, POLR1HASP, C1orf220, MIR99AHG, MMP23A, LRRK2-DT, SNX10-AS1, LINC00115, ENSG00000226239, ENSG00000229591, FEZF1-AS1, ENSG00000230650, DCST1-AS1, ENSG00000232234, LINC00472, ENSG00000233461, ENSG00000233937, LRRFIP1P1, SNHG3, DUBR, LINC01011, ENSG00000244953, ENSG00000250978, ANKRD20A17P, ATP5PBP5, ENSG00000255389, MT1JP, ENSG00000257002, LINC02289, ENSG00000259275, NCAPGP2, LINC00622, LINC02555, ENSG00000261026, ENSG00000261136, LINC02185, ENSG00000263235, ENSG00000267137, LINC01082, ENSG00000269399, ENSG00000269918, ENSG00000269984, ENSG00000270557, ENSG00000270933, ENSG00000272037, KLHDC7B-DT, ENSG00000272848, ENSG00000274460, ENSG00000275993, LINC00624, ENSG00000279089, ENSG00000279208, ENSG00000279312, ENSG00000279529, ENSG00000279789, ENSG00000280067, BLACAT1, ENSG00000285979

In [ ]:
##################################
# PASO 3: Resumen final
##################################
separador = '=' * 60
print(f"\n{separador}")
print("  ARCHIVOS GENERADOS")
print(separador)

for nombre, res in RESULTADOS.items():
    n = len(res['genes'])
    print(f"\n{nombre}  ({n:,} genes)")
    print(f"  DGE : {nombre}_DGE.csv   — matriz Pearson norm. [{n}×{n}]")
    print(f"  DBI : {nombre}_DBI.csv   — similitud semántica GO  [{n}×{n}]")